#### Prompt Templates

In [ ]:
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

In [ ]:
examples = [
    {
        "question": "Who held office longer as Prime Minister, Pierre Trudeau or Wilfrid Laurier?",
        "answer": """Does this question require additional questions: Yes.
Additional Question: How long was Pierre Trudeau Prime Minister of Canada?
Intermediate Answer: Pierre Trudeau served as Prime Minister for approximately 15 years and 164 days.
Additional Question: How long was Wilfrid Laurier Prime Minister of Canada?
Intermediate Answer: Wilfrid Laurier served as Prime Minister for 15 years and 86 days.
The final answer is: Pierre Trudeau""",
    },
    {
        "question": "When was the founder of Shopify born?",
        "answer": """Does this question require additional questions: Yes.
Additional Question: Who is the founder of Shopify?
Intermediate Answer: Shopify was co-founded by Tobias Lütke.
Additional Question: When was Tobias Lütke born?
Intermediate Answer: Tobias Lütke was born on July 16, 1980.
The final answer is: July 16, 1980""",
    },
    {
        "question": "Who was the reigning monarch when the author of Anne of Green Gables was born?",
        "answer": """Does this question require additional questions: Yes.
Additional Question: Who is the author of Anne of Green Gables?
Intermediate Answer: The author of Anne of Green Gables is Lucy Maud Montgomery.
Additional Question: When was Lucy Maud Montgomery born?
Intermediate Answer: Lucy Maud Montgomery was born on November 30, 1874.
Additional Question: Who was the British/Canadian monarch in 1874?
Intermediate Answer: The monarch in 1874 was Queen Victoria.
The final answer is: Queen Victoria""",
    },
    {
        "question": "Are the directors of Incendies and Water from the same country?",
        "answer": """Does this question require additional questions: Yes.
Additional Question: Who is the director of Incendies?
Intermediate Answer: The director of Incendies is Denis Villeneuve.
Additional Question: Which country is Denis Villeneuve from?
Intermediate Answer: Denis Villeneuve is from Canada.
Additional Question: Who is the director of Water?
Intermediate Answer: The director of Water is Deepa Mehta.
Additional Question: Which country is Deepa Mehta from?
Intermediate Answer: Deepa Mehta is Canadian (Indian-born Canadian).
The final answer is: Yes""",
    },
]

In [ ]:
example_prompt = PromptTemplate.from_template("Question: {question}\nAnswer: {answer}")
print(example_prompt.format(question=examples[0]["question"], answer=examples[0]["answer"]))

In [ ]:
few_shot_prompt = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question:\n{question}\nAnswer:",
    input_variables=["question"],
)

In [ ]:
question = "How old was Bill Gates when Google was founded?"

# Generate the final prompt
final_prompt = few_shot_prompt.format(question=question)

In [ ]:
print(final_prompt)

#### Getting Prompts from Hub

In [ ]:
from langchainhub import Client

client = Client()
prompt = client.pull("rlm/rag-prompt")

In [ ]:
print(prompt)

#### Prompt Caching 
In LangChain, there are two distinct types of caching often referred to as "prompt caching":

- 1 Vendor-Native Prompt Caching (Anthropic, OpenAI, etc.) — Reduces LLM latency/costs by passing structural cache markers to provider models.

- 2 LangChain Application-Side Caching — Saves complete LLM outputs locally (e.g., in Redis, SQLite, or In-Memory) so identical prompts skip the LLM API call entirely.

##### Method 1: Vendor-Native Caching in LangChain
A. Anthropic (Claude)
Anthropic uses explicit cache_control blocks. You can attach cache_control directly to system prompts or use LangChain's built-in AnthropicPromptCachingMiddleware.

B. OpenAI applies prompt caching automatically for prompts over 1,024 tokens. No code changes are required in LangChain beyond structuring your system messages first:

Option 1: Explicit cache_control in Prompt Templates

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate

llm = ChatAnthropic(model="claude-3-5-sonnet-20241022")

# Massive document / system prompt block to cache
large_system_prompt = "You are a legal assistant... [1,000+ words of text]"

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            [
                {
                    "type": "text",
                    "text": large_system_prompt,
                    # Add cache control block to mark the end of the static prefix
                    "cache_control": {"type": "ephemeral"},
                }
            ],
        ),
        ("human", "{question}"),
    ]
)

chain = prompt | llm
response = chain.invoke({"question": "Summarize section 4."})

# Using middleware
from langchain_anthropic import ChatAnthropic
from langchain_anthropic.middleware import AnthropicPromptCachingMiddleware

# Wrap model with prompt caching middleware
llm = ChatAnthropic(
    model="claude-3-5-sonnet-20241022",
    middleware=[AnthropicPromptCachingMiddleware(ttl="5m")],
)

##### Method 2: LangChain Application-Side Response Caching

In [ ]:
from langchain.globals import set_llm_cache
from langchain_community.cache import InMemoryCache
from langchain_openai import ChatOpenAI

# Enable in-memory cache globally
set_llm_cache(InMemoryCache())

llm = ChatOpenAI(model="gpt-4o-mini")

# 1st call: Hits the LLM API
response1 = llm.invoke("What is the capital of Canada?")

# 2nd call: Returned instantly from local cache (0 API cost, 0 latency)
response2 = llm.invoke("What is the capital of Canada?")